In [5]:
import sys
sys.path.append('..')
from utils.prompts import render
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.router import pick_model

In [6]:
with open('..\\data\\Scenarios.txt', 'r') as file:
    scenarios = file.read().split('\n\n')
    
# for scenario in scenarios:
#     print("Scenario:")
#     print(scenario)
#     print("-" * 40)
    
# CoT auto-routes to reasoning model
reasoning_model = pick_model('openai', 'cot')
client_reasoning = LLMClient('openai', reasoning_model)

print(f'Using reasoning model: {reasoning_model}')

instructions = """
You are an emergency response coordinator. Analyze this crisis scenario and provide:
1. Assessment of the situation
2. Recommended priority actions
3. Resource allocation decisions

Think through your reasoning step by step.
"""

# Experiment Configuration
# Chaos Mode: 3 runs at temp 1.0 | Safe Mode: 1 run at temp 0.0
experiment_configs = [
    {"temp": 0.0, "runs": 1, "label": "Safe Mode"},
    {"temp": 1.0, "runs": 3, "label": "Chaos Mode"}
]

chaos_outputs = []
safe_outputs = []

for i, scenario in enumerate(scenarios):
    print(f"\n### PROCESSING SCENARIO {i+1} ###")
    
    # Render the base prompt once per scenario
    prompt_text, spec = render(
        'cot_reasoning.v1',
        role="crisis-analysis assistant",
        problem=scenario
    )
    
    full_prompt = f"text: {prompt_text}\ninstruction: {instructions}"
    messages = [{'role': 'user', 'content': full_prompt}]

    # Nested loop for the experiment
    for config in experiment_configs:
        current_temp = config["temp"]
        num_runs = config["runs"]
        
        print(f"\n--- Starting {config['label']} (Temp: {current_temp}) ---")
        
        for run in range(num_runs):
            print(f"Run {run + 1}/{num_runs}...")
            
            response = client_reasoning.chat(
                messages, 
                temperature=current_temp, 
                max_tokens=spec.max_tokens
            )
            
            print(f"CoT Response (Run {run + 1}):")
            print('=' * 80)
            print(response['text'])
            print('=' * 80)
            
            log_llm_call('openai', reasoning_model, f'cot_{config["label"]}', response['latency_ms'], response['usage'])
            
            if config["label"] == "Safe Mode":
                safe_outputs.append(response['text'])
            else:
                chaos_outputs.append(response['text'])

Using reasoning model: o3-mini

### PROCESSING SCENARIO 1 ###

--- Starting Safe Mode (Temp: 0.0) ---
Run 1/1...
CoT Response (Run 1):
Reasoning Steps:
1. Assess immediate life threats: uncle’s drowning risk vs. diabetic patient’s critical insulin need.
2. Identify resource constraints: blocked road, limited access, and additional needs (40 hungry people).
3. Prioritize actions based on imminent danger and health deterioration.
4. Allocate resources to concurrently address water rescue, emergency healthcare, and sustenance.

Answer:
1. Assessment of the Situation:
 • The landslide has blocked the access road, complicating rescue and relief operations.
 • There are two immediate life threats: the uncle at risk of drowning due to rising water, and a diabetic patient in need of insulin.
 • Additionally, there is a group of 40 people suffering from hunger, which could lead to secondary health and morale issues if not addressed promptly.

2. Recommended Priority Actions:
 • Immediate Life-s

In [7]:
analysis_prompt = f"""
SAFE OUTPUTS:
{chr(10).join(safe_outputs)}

CHAOS OUTPUTS:
{chr(10).join(chaos_outputs)}

write a comment explaining specifically how the Chaos output drifted or hallucinated compared to the Safe output.
"""

analysis = client_reasoning.chat(
    [{'role': 'user', 'content': analysis_prompt}],
    temperature=0.0,
    max_tokens=300
)

print("\n=== STABILITY COMMENT ===")
print(analysis['text'])

compare_instructions = """
write a comment explaining specifically how the Chaos output drifted or hallucinated compared to the Safe output.
"""

compare_prompt_text, spec = render(
    'zero_shot.v1',
    role='evaluation assistant',
    instruction=compare_instructions,
    constraints='Return only the short comment',
    format='Sentence'
)

messages = [{'role': 'user', 'content': f"{compare_prompt_text}\n\nText: {analysis_prompt}"}]

model = pick_model('openai', 'general')
client = LLMClient('openai', model)

response = client.chat(messages, temperature=0.0)
print(response['text'])

log_llm_call('openai', reasoning_model, 'zero_shot', response['latency_ms'], response['usage'])


=== STABILITY COMMENT ===

The Chaos output drifted by presenting a less structured approach to prioritizing immediate life threats, failing to emphasize the need for concurrent resource allocation and detailed logistical planning, which were clearly outlined in the Safe output.


In [8]:
# print(safe_outputs)
# print(f"\n{chaos_outputs}\n")